# Modelo y validación

Entrena un Random Forest de clasificación que predice el nivel de ocupación (0 a 3)
a partir de las características de cada ventana, y mide lo bien que acierta en datos
que no ha visto.

Es **aprendizaje supervisado**: se le dan muchos ejemplos de los que ya sabemos la
respuesta (cada ventana de 3 minutos junto al `busy_level` que marcó una persona) y el
modelo aprende qué combinaciones de sensores corresponden a cada nivel. Después,
en directo, predice el nivel sin que nadie lo marque.

El proceso completo:

| Paso | Qué se hace | Por qué |
| --- | --- | --- |
| 0 | Limpiar los datos y convertirlos en ventanas | El modelo necesita una fila de números por ejemplo |
| 1 | Separar características (X) y respuesta (y) | X es lo que mira el modelo; y es lo que tiene que acertar |
| 2 | Agrupar las ventanas en bloques | Para que la validación no haga trampa con ventanas casi iguales |
| 3 | Entrenar y probar varias veces (validación cruzada) | Saber cómo acertará con datos que no ha visto |
| 4 | Calcular las métricas | Ver dónde acierta y dónde se equivoca |
| 5 | Entrenar el modelo final con todo | El que se usa en directo |
| 6 | Guardarlo | Para que `predict.py` lo cargue |

Es el mismo código que `src/occupancy/train.py`, separado en pasos para verlo con calma.

Diapositivas: clase 7, 63 (aprendizaje supervisado), 65 (clasificación),
67–69 (Random Forest) y 53–54 (muestreo estratificado).

**Aviso:** con los datos simulados los resultados saldrán muy buenos porque las
relaciones entre sensores y ocupación son inventadas. Este notebook sirve para entender
y tener listo el proceso; los resultados de verdad salen con datos reales
(`python -m occupancy.train`).

## Paso 0: de lecturas a ventanas

El modelo no trabaja con lecturas sueltas (una cada 10 s), sino con **ventanas**: trozos
de 3 minutos que se resumen en una fila de números (media de cada sensor, ruido máximo,
cuánto ha subido el CO2, actividad de los PIR...). Se calcula una ventana cada 30 s.

Cada ventana es **un ejemplo** para el modelo: sus características y, como respuesta,
el `busy_level` más frecuente durante esos 3 minutos. Se usan exactamente las mismas
funciones que en la predicción en directo, para que el modelo vea siempre los datos
calculados de la misma forma.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml

# Buscar la carpeta del repositorio (la que tiene config.yaml), suba desde donde
# se abra el notebook, y añadir src/ para poder importar el paquete occupancy
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("No se encuentra config.yaml: abre el notebook dentro del repositorio")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# Las mismas funciones que usan train.py y predict.py
from occupancy.cleaning import clean            # fase 1: limpieza
from occupancy.integration import merge_all     # fase 2: todo en una tabla
from occupancy.features import build_windows    # fase 4: ventanas y características

# Toda la configuración (límites, tamaño de ventana...) sale de config.yaml
with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Datos simulados, en el mismo formato que devuelve la API
with open(ROOT / "data" / "simulated" / "readings.json", encoding="utf-8") as f:
    data = json.load(f)["data"]

# Fase 1: limpiar cada sensor (duplicados, rango, picos, huecos, suavizado)
tables, pir, labels = clean(data, config["cleaning"])
# Fase 2: unir sensores, PIR y etiqueta en una tabla con una fila cada 10 s
df = merge_all(tables, pir, labels)
# Fase 4: una ventana de 3 min cada 30 s, cada una resumida en una fila.
# Los sensores salen de config.yaml, igual que en train.py y predict.py, para que
# las columnas estén siempre en el mismo orden.
windows = build_windows(df, config["windows"]["sensors"], config["windows"],
                        config["cleaning"]["interval_seconds"],
                        label_column=config["cleaning"]["label"]["sensor"])

# Cuántas ventanas hay (filas x columnas) y cuántas de cada nivel.
# Si un nivel tiene muy pocas, el modelo lo aprenderá peor.
print("Ventanas:", windows.shape)
print(windows["label"].value_counts().sort_index())

Ventanas: (345, 15)
label
0    201
1     22
2     45
3     77
Name: count, dtype: int64


## Paso 1: separar características (X) y respuesta (y)

- **X** (características): los 14 números de cada ventana. Es lo único que el modelo
  puede mirar, tanto al entrenar como en directo.
- **y** (respuesta o etiqueta): el nivel real de cada ventana. Al entrenar, el modelo
  compara sus predicciones con `y` para aprender; en directo no la tiene.

In [2]:
X = windows.drop(columns="label")   # características de cada ventana (lo que ve el modelo)
y = windows["label"]                # nivel real (0 a 3), lo que el modelo debe aprender

# Misma cantidad de filas en X y en y: una respuesta por ventana
print("X:", X.shape, " y:", y.shape)
X.head()

X: (345, 14)  y: (345,)


,pressure_mean,humidity_mean,temperature_mean,co2Equivalent_mean,voc_mean,iaq_mean,mic_vol_mean,wifiPackets_mean,bleAdverts_mean,co2Equivalent_change,mic_vol_max,mic_vol_std,pir_mean,pir_max
timestamp,,,,,,,,,,,,,,
2026-10-20 09:03:00,1013.043333,45.816667,21.115000,530.497222,0.665065,169.416667,-32.814722,81.500000,42.611111,128.033333,-31.380000,0.995140,97.407407,100.0
2026-10-20 09:03:30,1013.054444,45.921667,21.148333,552.311111,0.662519,169.016667,-32.978148,78.944444,40.833333,131.666667,-31.736667,0.916698,96.851852,100.0
2026-10-20 09:04:00,1013.031111,45.976111,21.170000,573.322222,0.670759,167.716667,-33.240370,78.722222,39.833333,123.800000,-31.736667,0.888531,97.037037,100.0
2026-10-20 09:04:30,1012.985556,46.020000,21.186111,594.294444,0.674907,166.127778,-33.412222,80.000000,40.722222,125.900000,-31.736667,0.825850,97.222222,100.0
2026-10-20 09:05:00,1012.917222,46.075000,21.197222,616.429630,0.671185,166.294444,-33.355185,79.555556,41.611111,137.700000,-31.736667,0.766195,97.222222,100.0


## Paso 2: grupos para la validación

Para saber si el modelo sirve hay que probarlo con ventanas que **no ha visto al entrenar**.
El problema: una ventana se calcula cada 30 s pero dura 3 min, así que dos ventanas
seguidas comparten 2,5 minutos y son casi iguales. Si una va a entrenar y su vecina a
probar, el modelo "se sabe la respuesta" y la exactitud sale demasiado buena.

Por eso se agrupan en **bloques de 15 minutos** y cada bloque entero va o a entrenar o
a probar, nunca repartido. Con datos reales de varias sesiones, el grupo es la sesión
entera (`train.py` lo hace así), que es todavía más exigente: prueba si el modelo
funciona otro día, con otra gente y otro tiempo.

In [3]:
# Redondear la hora de cada ventana hacia abajo a 15 min: 09:07 y 09:14 -> bloque 09:00
grupos = windows.index.floor("15min").astype(str)

print("Bloques:", grupos.nunique())

Bloques: 12


## Paso 3: entrenar y probar en cada partición (validación cruzada)

**Qué es un Random Forest.** Es un conjunto de árboles de decisión. Cada árbol aprende
reglas del tipo "si `mic_vol_mean` > -32 y `co2Equivalent_change` > 5, entonces nivel 3".
Cada uno se entrena con una muestra al azar de las ventanas y, en cada pregunta, mira
solo algunas características al azar, así que cada árbol es un poco distinto. Para
predecir, **todos votan**: el nivel más votado es la predicción y el porcentaje de
votos es la probabilidad (la confianza que se ve en `predict.py`). Muchos árboles
distintos votando se equivocan menos que uno solo.

**Por qué este modelo:** funciona bien con pocos datos, no necesita normalizar (le da
igual que la presión sea 1028 y el VOC 0,6) y dice qué características usa más.

**Validación cruzada.** Los bloques se reparten en 4 partes, manteniendo en cada una
una proporción parecida de niveles (muestreo estratificado, diapositivas 53–54).
Se hacen 4 rondas: en cada una se entrena con 3 partes y se prueba con la que queda.
Al final, cada ventana ha sido predicha una vez por un modelo que no la había visto,
y con todas esas predicciones se calculan las métricas del paso 4.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold

# 4 partes; mismos bloques siempre juntos (Group) y proporción de niveles parecida
# en cada parte (Stratified). shuffle + random_state: reparto al azar pero repetible.
particiones = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
reales, predichos = [], []

for entreno, prueba in particiones.split(X, y, grupos):
    # 'entreno' y 'prueba' son las posiciones de las ventanas de cada lado

    # Huecos (un sensor sin datos en la ventana): se rellenan con la mediana.
    # La mediana se calcula SOLO con las ventanas de entrenamiento: en directo
    # tampoco conoceremos los datos futuros, y así la prueba es honesta.
    relleno = X.iloc[entreno].median()

    modelo = RandomForestClassifier(
        n_estimators=200,          # 200 árboles votando: más estable que pocos
        class_weight="balanced",   # los niveles con pocas ventanas (aquí el 1) pesan
                                   # más, para que el modelo no los ignore
        random_state=42,           # mismo resultado cada vez que se ejecuta
    )
    # Aprender: el modelo ve X e y de las ventanas de entrenamiento
    modelo.fit(X.iloc[entreno].fillna(relleno), y.iloc[entreno])

    # Probar: predice las ventanas que no ha visto y guardamos real y predicho
    reales.extend(y.iloc[prueba])
    predichos.extend(modelo.predict(X.iloc[prueba].fillna(relleno)))

reales, predichos = np.array(reales), np.array(predichos)
print("Ventanas predichas sin haberlas visto:", len(predichos))

Ventanas predichas sin haberlas visto: 345


## Paso 4: métricas

- **Exactitud**: porcentaje de ventanas en las que acierta el nivel exacto.
- **Acierto con margen de un nivel**: confundir 2 con 3 es menos grave que 0 con 3;
  esto cuenta como acierto equivocarse por uno.
- **Matriz de confusión**: cada fila es el nivel real y cada columna el predicho. Lo
  ideal es todo en la diagonal; los números fuera de ella dicen qué niveles confunde.
- **Por nivel** (`classification_report`):
  - *precision*: de las veces que dijo "nivel 2", cuántas era de verdad 2;
  - *recall*: de las ventanas que eran nivel 2, cuántas encontró;
  - *f1*: combina las dos; *support*: cuántas ventanas había de ese nivel.

Hay que mirar los niveles por separado: con muchos 0, un modelo que dijera siempre 0
tendría una exactitud alta y aun así sería inútil.

In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

niveles = [0, 1, 2, 3]
print("Exactitud:", round(accuracy_score(reales, predichos), 3))
# |real - predicho| <= 1 -> acierto exacto o equivocado en un solo nivel
print("Acierto con margen de un nivel:", round(np.mean(np.abs(reales - predichos) <= 1), 3))
print()
# Filas = nivel real, columnas = nivel predicho
print(pd.DataFrame(confusion_matrix(reales, predichos, labels=niveles),
                   index=[f"real {n}" for n in niveles],
                   columns=[f"pred {n}" for n in niveles]))
print()
# precision, recall y f1 de cada nivel (zero_division=0: si un nivel no aparece, 0 en vez de aviso)
print(classification_report(reales, predichos, labels=niveles, zero_division=0))

Exactitud: 0.98
Acierto con margen de un nivel: 1.0

        pred 0  pred 1  pred 2  pred 3
real 0     200       1       0       0
real 1       1      21       0       0
real 2       0       2      41       2
real 3       0       0       1      76

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       201
           1       0.88      0.95      0.91        22
           2       0.98      0.91      0.94        45
           3       0.97      0.99      0.98        77

    accuracy                           0.98       345
   macro avg       0.96      0.96      0.96       345
weighted avg       0.98      0.98      0.98       345



## Paso 5: entrenar el modelo final y ver qué características usa más

Los 4 modelos del paso 3 solo servían para **medir**; se descartan. El modelo que se usa
en directo se entrena con **todas** las ventanas, porque cuantos más ejemplos, mejor
aprende. Su acierto esperado es el que dieron las métricas del paso 4.

La **importancia** dice cuánto usa el bosque cada característica para separar los
niveles (suman 1). Las que salen casi a cero son candidatas a quitarse (reducción,
diapositivas 34–46), comprobando que la exactitud no baja sin ellas. Ojo: una
característica muy importante no siempre es buena; con datos reales, si la presión
(que depende del tiempo atmosférico) sale la primera, el modelo puede estar
aprendiendo el día y no la ocupación.

In [6]:
# Relleno con la mediana de todas las ventanas: es el que se usará en directo
relleno = X.median()
# Mismos parámetros que en la validación, para que las métricas sean representativas
modelo_final = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
modelo_final.fit(X.fillna(relleno), y)

# Importancia de cada característica, de mayor a menor
pd.Series(modelo_final.feature_importances_, index=X.columns).sort_values(ascending=False).round(3)

mic_vol_max             0.185
iaq_mean                0.147
wifiPackets_mean        0.132
mic_vol_mean            0.125
voc_mean                0.115
pir_mean                0.111
bleAdverts_mean         0.104
co2Equivalent_mean      0.018
pir_max                 0.018
temperature_mean        0.012
humidity_mean           0.012
co2Equivalent_change    0.011
mic_vol_std             0.009
pressure_mean           0.001
dtype: float64

## Paso 6: guardar el modelo final

Se guarda con todo lo que `predict.py` necesita para usarlo en directo:

| Clave | Para qué |
| --- | --- |
| `model` | El Random Forest entrenado |
| `features` | Nombres y orden de las columnas: en directo hay que darlas en el mismo orden |
| `fill_values` | Mediana de cada característica, para rellenar si un sensor falla en directo |
| `feature_min`, `feature_max` | Rango visto al entrenar: si un valor en directo se sale, la predicción se marca como no fiable (`<- outside`) |

**Cuidado:** guardar sobrescribe `models/model.joblib`, que es el modelo que usa
`predict.py`. Este notebook entrena con los datos simulados, así que por defecto **no
guarda** (`GUARDAR = False`) para no pisar el modelo entrenado con datos reales.

In [7]:
GUARDAR = False   # True para sobrescribir models/model.joblib con este modelo (simulado)

modelo_guardado = {
    "model": modelo_final,
    "features": list(X.columns),   # orden de las columnas
    "fill_values": relleno,        # para rellenar huecos en directo
    "feature_min": X.min(),        # rango visto al entrenar
    "feature_max": X.max(),
}
ruta = ROOT / config["model"]["path"]
if GUARDAR:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(modelo_guardado, ruta)   # joblib guarda el diccionario entero en un archivo
    print("Modelo guardado en", ruta)
else:
    print("No se ha guardado (GUARDAR = False). El modelo de", ruta, "no cambia.")

No se ha guardado (GUARDAR = False). El modelo de C:\Users\danib\Desktop\ERASMUS\IOT\loudness-classifier\models\model.joblib no cambia.
